In [ ]:
################################################
# New style 
################################################
import sys
import os

workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

from myPythonTools.Utils import utils as uti
from myPythonTools.Utils import numerical_utils as nuti
from myPythonTools.Utils import regridding_utils as ruti
from myPythonTools.Utils import AveragingUtils as Av
from myPythonTools.Utils import validation_data as Val
from myPythonTools.Utils import PlotUtil as Pu
from myPythonTools.Plotting import LatLonMaps as LL 
from myPythonTools.Plotting import utils as puti
from myPythonTools.CASutils import filter_utils as fu 

from PyRegridding.Utils import GridUtils as GrU
from PyRegridding.Utils import MakePressures as MkP
from PyRegridding.Drivers import RegridField as RgF


# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.tri as tri

# Cartopy for pretty maps
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Some other useful packages 
import importlib
import copy
import time
import cftime

importlib.reload( uti )
importlib.reload( puti )
importlib.reload( nuti )
importlib.reload( ruti )
importlib.reload( Pu )
importlib.reload(Av)
importlib.reload(Val)
#importlib.reload(vAB)
importlib.reload(MkP)
importlib.reload(RgF)


In [ ]:
class AttrDict(dict):
    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")

    def __setattr__(self, key, value):
        self[key] = value

    def __delattr__(self, key):
        try:
            del self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")


In [ ]:
case = f'c153_topfix_ne240pg3_FMTHIST_xic_x02'
base = f'/glade/derecho/scratch/juliob/archive/{case1}/atm/fv1x1/{case1}.cam.h1i'



In [ ]:
year = 2004
dates=[]  #[f"{year:04d}-{month:02d}-{day:02d}-{hour*3_600:05d}"]
for m in np.arange( 12 ):
    month=m+1
    date_=f"{year:04d}-{month:02d}"
    dates.append( date_ )

file = [] 
for date in dates:
    f1 = f'{base1}.{date}.nc'
    files1.append( f1 )
    #f2 = f'{base2}.{date}.nc'
    #files2.append( f2 )





In [ ]:
#casedic={'basedir':,'case':,'Hkey':}

cases=[]
basedir=f'/glade/derecho/scratch/juliob/archive'
case, Hkey = 'CTL_fmt_c64116'  , 'c'
cases.append( {'basedir':basedir,'case':case  , 'Hkey':Hkey} )
case, Hkey = 'CTL_fmt_c64144'  , 'c'
cases.append( {'basedir':basedir,'case':case  , 'Hkey':Hkey} )
case, Hkey = 'CTLre_fmt_c6477' , 'c'
cases.append( {'basedir':basedir,'case':case  , 'Hkey':Hkey} )

basedir=f'/glade/derecho/scratch/hannay/archive'
case, Hkey = 'f.e30_cam6_4_097.FHISTC_MTso.ne30.001' , 'yx'
cases.append( {'basedir':basedir,'case':case  , 'Hkey':Hkey} )
case, Hkey = 'f.e30_alpha08b.FHISTC_MTso.ne30_t232_wgx3.315' , 'yx'
cases.append( {'basedir':basedir,'case':case  , 'Hkey':Hkey} )
case, Hkey = 'b.e30_alpha08a.B1850C_MTso.ne30_t232_wgx3.280','yx'
cases.append( {'basedir':basedir,'case':case  , 'Hkey':Hkey} )

basedir=f'/glade/derecho/scratch/tilmes/archive'
case, Hkey = 'b.e30_alpha07c_cesm.B1850C_MTt4s.ne30_t232_wgx3.251.001' , 'yx_proc'
cases.append( {'basedir':basedir,'case':case  , 'Hkey':Hkey} )

basedir=f'/glade/derecho/scratch/shawnh/archive'
case, Hkey = 'b.e30_beta06_megan.B1850C_MTt4s.ne30.004' , 'yx'
#cases.append( {'basedir':basedir,'case':case  , 'Hkey':Hkey} )



In [ ]:
%%time
# Construct structure to look at Q and CH4 in multiple cases
tic_total_0 = time.perf_counter()

mos = [99,] # np.arange( start=1,stop=2)   #13)
Al=[]
for c in [cases[0]]:
    basedir,case,Hkey = c['basedir'],c['case'],c['Hkey']
    filep = f"{basedir}/{case}/atm/hist/{case}.cam.h0a.*-*.nc"
    print( f"Begin Read {filep} ")
    X=xr.open_mfdataset( filep ,data_vars='different', coords='different', compat='no_conflicts' )
    print( f"Finsihed Read {filep} ")
    tic_total_1 = time.perf_counter()
    pTime = f"Total reading took  {tic_total_1 - tic_total_0:0.4f} seconds"
    print( pTime )
    X2=X[['lat','lon','PS','Q','CH4']]
    X2 = X2.groupby("time.year").mean(dim="time")     #.compute()
    X2['lat'] = X2['lat'].isel(year=0)
    X2['lon'] = X2['lon'].isel(year=0)
    X2 = X2.rename({"year": "time"})
    print( f"Finsihed Group by year ")
    print( f"Size of Q: {np.shape(X2.Q)}" )
    tic_total_2 = time.perf_counter()
    pTime = f"Group by year took  {tic_total_2 - tic_total_1:0.4f} seconds"
    print( pTime )
        #break
